# 1-step average reward actor critic with normal distribution policy

Pendulum source code: https://github.com/openai/gym/blob/master/gym/envs/classic_control/pendulum.py </br>
Pendulum documentation: https://github.com/openai/gym/wiki/Pendulum-v0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gym
import time
from tqdm import tqdm
from itertools import product
import rl.features.tile_coding as tc
from rl.animations.video import write_video, show_video, simulate_episode

np.random.seed(42)

# Cart Pole environment

<img src="https://blog.paperspace.com/content/images/2018/12/freegifmaker.me_2dR21.gif">

In [ ]:
env = gym.make("Pendulum-v0")


class Pendulum(gym.envs.classic_control.PendulumEnv):
    
    def _get_obs(self):
        theta, thetadot = self.state
        return theta, thetadot
    
    
# env = Pendulum()

In [ ]:
print(env.observation_space)
print(env.action_space)

In [ ]:
# quick experiment to see what the maximum velocities are that the environment really returns
# cos_angle, sin_angle, vel = obs
data = []
for i in range(100):
    obs = env.reset()
    data.append(obs)
    terminal = False
    for i in range(200):
        obs, reward, terminal, _ = env.step(env.action_space.sample())
        data.append(obs)

pd.DataFrame(data, columns=['cos_angle', 'sin_angle', 'velocity']).describe()

# Agent

In [ ]:
class Agent:
    
    def __init__(self, env, alpha_critic=0.01, alpha_actor_m=0.01, alpha_actor_s=0.01, alpha_avg_reward=0.01, num_tilings=8, num_tiles=8, iht_size=4096):
        self.action_space = env.action_space
        self.alpha_avg_reward = alpha_avg_reward
        self.alpha_critic = alpha_critic
        self.alpha_actor_m = alpha_actor_m
        self.alpha_actor_s = alpha_actor_s
        
        self.last_action = None
        self.last_softmax_probs = None
        self.last_state = None
        self.last_features = None
        
        self.num_tilings = num_tilings
        self.num_tiles = num_tiles
        self.iht_size = iht_size
        self.iht = tc.IHT(iht_size)
        
        self.critic_w = np.zeros(self.iht_size)  # critic weights
        self.actor_w_m = np.zeros(self.iht_size)  # actor weight for mu
        self.actor_w_s = np.zeros(self.iht_size)  # actor weights for sigma
        
        self.avg_reward = 0.
        self.epsilon = 0.0001  # np.finfo(float).eps
    
    def state_to_tiles(self, state):
        angle_cos, angle_sin, vel = state
        angle_cos_scaled = ((angle_cos + 1) / 2) * self.num_tiles
        angle_sin_scaled = ((angle_sin + 1) / 2) * self.num_tiles
        vel_scaled = ((vel + 8) / 16) * self.num_tiles
        floats = [angle_cos_scaled, angle_sin_scaled, vel_scaled]
        return np.array(tc.tiles(self.iht, self.num_tilings, floats))
    
    def select_action(self, mu, sigma):
#         mu = np.clip(mu, -2, 2)
        action = np.random.normal(loc=mu, scale=sigma, size=(1,))
        return action
    
    def agent_start(self, state):
        features = self.state_to_tiles(state)
        mu = self.get_actor_mu(features)
        sigma = self.get_actor_sigma(features)
        self.last_action = self.select_action(mu, sigma)
        self.last_state = state
        self.last_features = features
        return self.last_action
    
    def agent_step(self, reward, state, terminal=False):
        features = self.state_to_tiles(state)
        mu = self.get_actor_mu(features)
        sigma = self.get_actor_sigma(features)
        action = self.select_action(mu, sigma)
        # calculate delta
        delta = reward - self.avg_reward + self.get_critic_estimate(features) * (1 - terminal) - self.get_critic_estimate(self.last_features)
        # update average reward estimate
        self.avg_reward += self.alpha_avg_reward * delta
        # update critic weights
        self.critic_w[self.last_features] += self.alpha_critic * delta  # *1 for the gradient officially
        # update actor weights
        grad_mu, grad_sigma = self.get_gradients(self.last_features, self.last_action)
        if np.isnan(grad_mu[0]):
            print("delta, last_action, last_mu", delta, self.last_action, self.get_actor_mu(self.last_features))
        if np.isnan(grad_sigma[0]):
            print("delta, last_action, last_sigma", delta, self.last_action, self.get_actor_sigma(self.last_features))
        self.actor_w_m += self.alpha_actor_m * delta * grad_mu
        self.actor_w_s += self.alpha_actor_s * delta * grad_sigma
        # select next action
        self.last_action = action
        self.last_state = state
        self.last_features = features
    
    def get_critic_estimate(self, features):
        return self.critic_w[features].sum()
    
    def get_actor_mu(self, features):
        return 2 * np.tanh(self.actor_w_m[features].sum())
#         return self.actor_w_m[features].sum()
    
    def get_actor_sigma(self, features):
        return np.exp(self.actor_w_s[features].sum())
    
    def get_gradients(self, features, action):
        m = self.get_actor_mu(features)
        s = self.get_actor_sigma(features)
        feature_vector = np.zeros(self.iht_size)
        feature_vector[features] = 1
        grad_mu = (2 / (s**2 + self.epsilon)) * (action - m) * (1 - np.tanh(m)**2) * feature_vector
#         grad_mu = (1 / (s**2 + self.epsilon)) * (action - m) * feature_vector
        grad_sigma = ((action - m)**2 / (s**2 + self.epsilon) - 1) * feature_vector
        return np.clip(grad_mu, -1, 1), np.clip(grad_sigma, -1, 1)
    
    def state_to_action(self, state):
        return self.state_to_action_info(state)[0]
    
    def state_to_action_info(self, state):
        features = self.state_to_tiles(state)
        mu = self.get_actor_mu(features)
        sigma = self.get_actor_sigma(features)
        return self.select_action(mu, sigma), mu, sigma

# Helper functions

In [ ]:
def episode(env, agent):
    """Run one (training) episode.
    """
    last_observation = env.reset()
    terminal = False
    cumulative_reward = 0
    state_list = []
    agent.agent_start(last_observation)
    while not terminal:
        observation, reward, terminal, info = env.step(agent.last_action)
        cumulative_reward += reward
        state_list.append(observation)
        if info.get('TimeLimit.truncated'):
            # environment truncated the episode, therefore terminal is actually False
            agent.agent_step(reward, observation, False)
        else:
            agent.agent_step(reward, observation, terminal)
    return cumulative_reward, state_list


def run_experiment(env, agent, n_episodes=1000, title=None):
    reward_list = []
    state_lists = []
    pbar = tqdm(range(n_episodes))
    for i in pbar:
        if title:
            pbar.set_description(title)
        episode_reward, state_list = episode(env, agent)
        reward_list.append(episode_reward)
        state_lists.append(state_list)
    return reward_list, state_lists


def investigate_agent_actions(env, agent, n_steps=500):
    obs = env.reset()
    terminal = False
    i = 0
    data = []
    while not terminal and i < n_steps:
        action, m, s = agent.state_to_action_info(obs)
        data.append([action[0], m, s])
        obs, _, terminal, _ = env.step(action)
        i += 1
    return pd.DataFrame(data, columns=['action', 'mu', 'sigma'])

# Example episodes

In [ ]:
# random agent
simulate_episode(env, lambda x: env.action_space.sample(), max_steps=600, width=500, play_type='autoplay')

# Single agent training

In [ ]:
agent = Agent(env, alpha_critic=2**-14, alpha_actor_m=2**-16, alpha_actor_s=2**-19, alpha_avg_reward=2**-14)
rewards, observations = run_experiment(env, agent, n_episodes=100000)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

ax.set_title("Rolling mean of total reward per episode", fontsize=20)
ax.set_xlabel("Episode Number", fontsize=16)
ax.set_ylabel("Reward per Episode", fontsize=16)
ax.plot(pd.Series(rewards).rolling(window=100).mean());

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3))

df_choices = investigate_agent_actions(env, agent)
df_choices.hist(ax=axes);

In [ ]:
# trained agent
simulate_episode(env.env, agent.state_to_action, max_steps=600, width=400, play_type='controls')

# Start the experiment with hyperparameter grid search

In [ ]:
grid = {
    'alpha_critic': [2**-4, 2**-5],
    'alpha_actor': [2**-7, 2**-8],
    'alpha_avg_reward': [2**-6, 2**-7]
}


def get_hyperparameter_dict(grid):
    param_names = list(grid.keys())
    combinations = product(*[grid[name] for name in param_names])
    for values in combinations:
        yield dict(zip(param_names, values))

In [ ]:
results = dict()
for i, agent_setup in enumerate(get_hyperparameter_dict(grid)):
    agent = Agent(env, **agent_setup)
    rewards, observations = run_experiment(env, agent, n_episodes=1000, title='Agent Setup {}: {}'.format(i, agent_setup))
    observations = np.array([s for lst in observations for s in lst])
    results['run' + str(i)] = {'agent': agent, 'agent_setup': agent_setup, 'rewards': rewards, 'observations': observations}

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

ax.set_title("Rolling mean of total reward per episode", fontsize=20)
ax.set_xlabel("Episode Number", fontsize=16)
ax.set_ylabel("Reward per Episode", fontsize=16)
for run in results.keys():
    ax.plot(pd.Series(results[run]['rewards']).rolling(window=50).mean(), label=results[run]['agent_setup'])
ax.legend();

In [ ]:
# get the best agent based on the latest rewards
best_average_reward = -np.inf
for run in results.keys():
    final_average_reward = np.mean(results[run]['rewards'][-50:])
    if final_average_reward > best_average_reward:
        best_average_reward = final_average_reward
        best_agent = results[run]['agent']
        observations = results[run]['observations']

In [ ]:
# trained agent
simulate_episode(env.env, best_agent.state_to_action, max_steps=2000, width=600, play_type='controls')

# Visualizations

In [ ]:
states = pd.DataFrame(observations, columns=['cart_pos', 'cart_vel', 'pole_ang', 'pole_vel'])
cart_pos_linspace = np.linspace(-4.8, 4.8, 97)
cart_vel_linspace = np.linspace(-5, 5, 97)
pole_ang_linspace = np.linspace(-0.418, 0.418, 97)
pole_vel_linspace = np.linspace(-5, 5, 97)
states['cart_pos_bin'] = pd.IntervalIndex(pd.cut(states['cart_pos'], cart_pos_linspace)).mid
states['cart_vel_bin'] = pd.IntervalIndex(pd.cut(states['cart_vel'], cart_vel_linspace)).mid
states['pole_ang_bin'] = pd.IntervalIndex(pd.cut(states['pole_ang'], pole_ang_linspace)).mid
states['pole_vel_bin'] = pd.IntervalIndex(pd.cut(states['pole_vel'], pole_vel_linspace)).mid

In [ ]:
def state_visitation_heatmaps(bin_column1, bin_column2):
    state_visitation_counts = states.groupby([bin_column1, bin_column2]).size().unstack(fill_value=0)
    state_visitation_counts.index = np.round(state_visitation_counts.index, 3)
    state_visitation_counts.columns = np.round(state_visitation_counts.columns, 3)

    # create heatmap of visitation count
    fig, ax = plt.subplots(figsize=(18, 9))
    sns.heatmap(np.log(state_visitation_counts + 1).sort_index(ascending=False), ax=ax)
    ax.set_title("Natural log state visitation counts", fontsize=20)
    ax.set_xlabel(bin_column2, fontsize=16)
    ax.set_ylabel(bin_column1, fontsize=16);

In [ ]:
state_visitation_heatmaps('pole_ang_bin', 'cart_pos_bin')